# 06 · Scheduled & Deferred Messages

Two related-but-different delays:

| | Scheduled | Deferred |
|---|---|---|
| Who decides? | **Sender** | **Receiver** |
| When? | Before enqueue, sets `ScheduledEnqueueTime` | After receive, calls `DeferAsync` |
| Use case | Reminders, retries with backoff | "I can't process this yet — leave it for later, but don't let it block the queue." |
| How to retrieve | Auto-enqueued at the scheduled time | Must call `ReceiveDeferredMessagesAsync(sequenceNumber)` |


In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"

In [ ]:
#!import ../src/shared/Config.cs

In [ ]:
using Azure.Messaging.ServiceBus;

var client = new ServiceBusClient(Config.ConnectionString);
var sender = client.CreateSender(Config.QueueName);

## 1. Schedule a message for 10 seconds in the future

In [ ]:
var enqueueAt = DateTimeOffset.UtcNow.AddSeconds(10);

long seq = await sender.ScheduleMessageAsync(
    new ServiceBusMessage("future-message"),
    enqueueAt);

Console.WriteLine($"Scheduled (sequence={seq}) for {enqueueAt:HH:mm:ss}");

You can `CancelScheduledMessageAsync(seq)` before its time arrives.

We'll wait it out and receive it:

In [ ]:
var receiver = client.CreateReceiver(Config.QueueName);
var arrived = await receiver.ReceiveMessageAsync(TimeSpan.FromSeconds(15));
Console.WriteLine($"Received at {DateTimeOffset.UtcNow:HH:mm:ss}: {arrived?.Body}");
await receiver.CompleteMessageAsync(arrived);

## 2. Defer a message

Deferred messages **stay in the queue but are invisible** to normal receives. You must remember the `SequenceNumber` (typically by storing it in your own DB) to fetch them later.

In [ ]:
await sender.SendMessageAsync(new ServiceBusMessage("needs-extra-context") { MessageId = "ctx-1" });

var firstPass = await receiver.ReceiveMessageAsync(TimeSpan.FromSeconds(5));
long deferredSeq = firstPass.SequenceNumber;
await receiver.DeferMessageAsync(firstPass);
Console.WriteLine($"Deferred {firstPass.MessageId} (seq={deferredSeq})");

// A subsequent ReceiveMessageAsync would NOT return it:
var none = await receiver.ReceiveMessageAsync(TimeSpan.FromSeconds(2));
Console.WriteLine($"Normal receive saw: {(none?.Body ?? "(nothing)")}");

## 3. Retrieve the deferred message by sequence number

In [ ]:
var deferred = await receiver.ReceiveDeferredMessageAsync(deferredSeq);
Console.WriteLine($"Got back: {deferred.MessageId}  body={deferred.Body}");
await receiver.CompleteMessageAsync(deferred);

await receiver.DisposeAsync();
await sender.DisposeAsync();
await client.DisposeAsync();

Next: [`07-sessions.ipynb`](07-sessions.ipynb)